In [1]:
import pandas as pd
from pathlib import Path

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

raw_path = project_root / "data" / "raw" / "Dataset Capstone.xlsx"

processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

df_raw = pd.read_excel(
    raw_path,
    sheet_name="branches"
)

df_clean = df_raw.copy()

print("Jumlah data awal:", len(df_clean))
display(df_clean.head())

Jumlah data awal: 3


,branch_id,branch_name,city,address,opening_time,closing_time,phone,facilities,google_maps_url,instagram_url,gofood_url,grabfood_url,is_active,source_id,verification_status,data_type
0,BR001,KOPILOGI,GARUT,"Jl. Cikuray No.42, Regol, Kec. Garut Kota, Kab...",08:00:00,22:00:00,62 813 6786 6999,"wifi, ac, indoor, outdoor, toilet, parkir",https://maps.app.goo.gl/R479crDeXQ2gdv466,https://www.instagram.com/kopilogi?dlrf=MWVpcD...,https://gofood.link/a/zwzBrsq,NaN,True,SRC001,pending_pic,actual
1,BR002,WAROENG KOPILOGI,GARUT,"Jl. Guntur No.14, Paminggir, Kec. Garut Kota, ...",07:00:00,22:00:00,62 813 1222 0434,"wifi, ruang vip, karaoke, toilet, smoking area...",https://maps.app.goo.gl/DU4ZTnNwDtow4rj48,https://www.instagram.com/waroengkopilogi_garu...,https://gofood.link/a/BbAS6m1,NaN,True,SRC002,pending_pic,actual
2,BR003,KOPILOGI TASIK,TASIKMALAYA,"Jl. Dr. Sukarjo, Tawangsari, Kec. Tawang, Kota...",08:00:00,23:00:00,62 812 2228 1554,"wifi, ac, smoking area, mushola, toilet, slow ...",https://maps.app.goo.gl/tB6dk6wVaDYa7vzf8,https://www.instagram.com/kopilogi.tasik?cplk=...,https://gofood.link/a/LU2VaiJ,NaN,True,SRC003,pending_pic,actual


In [2]:
# Standardisasi nama cabang untuk tampilan
df_clean["branch_name"] = (
    df_clean["branch_name"]
    .astype("string")
    .str.strip()
    .str.title()
)

# Standardisasi kota
df_clean["city"] = (
    df_clean["city"]
    .astype("string")
    .str.strip()
    .str.title()
)

display(
    df_clean[
        ["branch_id", "branch_name", "city"]
    ]
)

,branch_id,branch_name,city
0,BR001,Kopilogi,Garut
1,BR002,Waroeng Kopilogi,Garut
2,BR003,Kopilogi Tasik,Tasikmalaya


In [3]:
from datetime import time

def format_time(value):
    if pd.isna(value):
        return pd.NA

    if isinstance(value, time):
        return value.strftime("%H:%M")

    parsed = pd.to_datetime(
        str(value),
        format="%H:%M:%S",
        errors="coerce"
    )

    if pd.isna(parsed):
        return pd.NA

    return parsed.strftime("%H:%M")

df_clean["opening_time"] = (
    df_clean["opening_time"].apply(format_time)
)

df_clean["closing_time"] = (
    df_clean["closing_time"].apply(format_time)
)

display(
    df_clean[
        ["branch_id", "opening_time", "closing_time"]
    ]
)

,branch_id,opening_time,closing_time
0,BR001,08:00,22:00
1,BR002,07:00,22:00
2,BR003,08:00,23:00


In [4]:
# Pertahankan nilai kosong sebagai unknown
df_clean["grabfood_url"] = (
    df_clean["grabfood_url"].astype("string")
)

print("Jumlah GrabFood kosong:")
print(df_clean["grabfood_url"].isna().sum())

display(
    df_clean[
        ["branch_id", "grabfood_url"]
    ]
)

Jumlah GrabFood kosong:
3


,branch_id,grabfood_url
0,BR001,<NA>
1,BR002,<NA>
2,BR003,<NA>


In [5]:
print("=== VALIDASI AKHIR DATA CLEANING CABANG ===")

# Memastikan jumlah record tidak berubah
assert len(df_clean) == len(df_raw), \
    "Jumlah record berubah!"

# Memastikan ID cabang unik
assert df_clean["branch_id"].is_unique, \
    "Terdapat branch_id duplikat!"

# Memastikan tidak ada ID kosong
assert df_clean["branch_id"].notna().all(), \
    "Terdapat branch_id kosong!"

# Memastikan format jam HH:mm
for col in ["opening_time", "closing_time"]:
    valid_time = df_clean[col].dropna().str.fullmatch(
        r"(?:[01]\d|2[0-3]):[0-5]\d"
    )
    assert valid_time.all(), f"Format {col} tidak valid!"

# Memastikan status verifikasi tidak berubah
assert (
    df_clean["verification_status"].tolist()
    == df_raw["verification_status"].tolist()
), "Status verifikasi berubah!"

print("Jumlah cabang:", len(df_clean))
print("Duplikasi ID:", df_clean["branch_id"].duplicated().sum())
print("GrabFood kosong:", df_clean["grabfood_url"].isna().sum())

print("\nValidasi struktur berhasil!")

=== VALIDASI AKHIR DATA CLEANING CABANG ===
Jumlah cabang: 3
Duplikasi ID: 0
GrabFood kosong: 3

Validasi struktur berhasil!


In [6]:
# Menentukan lokasi file hasil cleaning
csv_path = processed_dir / "branches_clean.csv"
xlsx_path = processed_dir / "branches_clean.xlsx"

# Menyimpan CSV
df_clean.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

# Menyimpan Excel
df_clean.to_excel(
    xlsx_path,
    index=False
)

print("Dataset berhasil disimpan:")
print(csv_path)
print(xlsx_path)

Dataset berhasil disimpan:
a:\ASAH DICODING\CAPSTONE PROJECT\kopilogi-data-science\data\processed\branches_clean.csv
a:\ASAH DICODING\CAPSTONE PROJECT\kopilogi-data-science\data\processed\branches_clean.xlsx


In [7]:
reports_dir = project_root / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

cleaning_log = pd.DataFrame([
    {
        "step": 1,
        "process": "Standardisasi nama cabang",
        "description": "Mengubah kapitalisasi nama cabang untuk tampilan",
        "result": "Nama cabang menggunakan format title case"
    },
    {
        "step": 2,
        "process": "Standardisasi nama kota",
        "description": "Mengubah kapitalisasi kota menjadi title case",
        "result": "Penulisan kota konsisten"
    },
    {
        "step": 3,
        "process": "Standardisasi jam operasional",
        "description": "Mengubah objek datetime.time menjadi format HH:mm",
        "result": "Jam siap diekspor ke CSV/JSON"
    },
    {
        "step": 4,
        "process": "Penanganan missing values",
        "description": "Mempertahankan GrabFood kosong karena belum ada sumber",
        "result": "Tidak dilakukan imputasi"
    },
    {
        "step": 5,
        "process": "Validasi",
        "description": "Memeriksa jumlah record, ID, format jam, dan status verifikasi",
        "result": "Validasi struktur berhasil"
    }
])

log_path = reports_dir / "branches_cleaning_log.csv"

cleaning_log.to_csv(
    log_path,
    index=False,
    encoding="utf-8-sig"
)

display(cleaning_log)

print("Cleaning log tersimpan:", log_path)

,step,process,description,result
0,1,Standardisasi nama cabang,Mengubah kapitalisasi nama cabang untuk tampilan,Nama cabang menggunakan format title case
1,2,Standardisasi nama kota,Mengubah kapitalisasi kota menjadi title case,Penulisan kota konsisten
2,3,Standardisasi jam operasional,Mengubah objek datetime.time menjadi format HH:mm,Jam siap diekspor ke CSV/JSON
3,4,Penanganan missing values,Mempertahankan GrabFood kosong karena belum ad...,Tidak dilakukan imputasi
4,5,Validasi,"Memeriksa jumlah record, ID, format jam, dan s...",Validasi struktur berhasil


Cleaning log tersimpan: a:\ASAH DICODING\CAPSTONE PROJECT\kopilogi-data-science\reports\branches_cleaning_log.csv
